# 🏪 YOLO-World Storefront Sign Detector via Hugging Face API

This notebook uses the Hugging Face `gradio_client` to detect storefront signs using the YOLO-World demo.


In [2]:
# Install required packages
!pip install gradio_client



[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [11]:
import os
import cv2
import pandas as pd
from gradio_client import Client
import time


In [8]:
# Setup paths
client = Client("https://stevengrove-yolo-world.hf.space/--replicas/x8hzw/")
input_dir = "imgs/RICHMOND"
output_dir = "imgs/RICHMOND_cropped"
os.makedirs(output_dir, exist_ok=True)


Loaded as API: https://stevengrove-yolo-world.hf.space/--replicas/x8hzw/ ✔


In [13]:
# Process images and collect results
results = []


for fname in os.listdir(input_dir):
    
    if not fname.lower().endswith(('.jpg')):
        continue
    

    image_path = os.path.join(input_dir, fname)

    try:
        time.sleep(5)
        result = client.predict(
            image_path,
            "store sign",
            3,     # max number of boxes
            0.3,   # score threshold
            0.4,   # nms threshold
            api_name="/partial"
        )

        boxes, scores, labels = result
        if boxes:
            img = cv2.imread(image_path)
            for i, box in enumerate(boxes):
                x1, y1, x2, y2 = map(int, box)
                cropped = img[y1:y2, x1:x2]
                crop_path = os.path.join(output_dir, f"{os.path.splitext(fname)[0]}_{i}.jpg")
                cv2.imwrite(crop_path, cropped)

            results.append({"name": fname, "has_sign": "yes", "num_boxes": len(boxes)})
        else:
            results.append({"name": fname, "has_sign": "no", "num_boxes": 0})

    except Exception as e:
        results.append({"name": fname, "has_sign": "error", "num_boxes": 0})
        print(f"Error processing {fname}: {e}")


Error processing 704147.jpg: Client error '429 Too Many Requests' for url 'https://stevengrove-yolo-world.hf.space/queue/join'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/429


KeyboardInterrupt: 

In [ ]:
# Save results to CSV
df = pd.DataFrame(results)
df.to_csv("yolo_world_sign_detection_results.csv", index=False)
df.head()
